# GridLock — Parking-Induced Congestion EDA

Reproducible analysis of the BTP parking-violation dataset.
Reads from `/kaggle/input/...`, writes checkpoints to `/kaggle/working`, charts to `/kaggle/working/eda_out`.

Cells run top-to-bottom on a fresh kernel: load+integrity → quality → univariate+severity → temporal → spatial → cross-dim → hidden patterns.

See `eda/findings/EDA_REPORT.md` for the executive summary.

In [ ]:
import os
os.makedirs("/kaggle/working/eda_out", exist_ok=True)
os.makedirs("/kaggle/working/derived", exist_ok=True)
print("output dirs ready")

## `00_load_integrity.py`

In [ ]:
import pandas as pd, numpy as np, hashlib, json, os
SRC = "/kaggle/input/datasets/kartikeysapkal/gridlock-round2-csv/jan to may police violation_anonymized791b166.csv"
df = pd.read_csv(SRC, dtype=str)            # read raw as str; we cast deliberately later
print("RAW shape:", df.shape)

# GUARD 1: expected row/col count (locally profiled = 298450 rows, 24 cols)
assert df.shape == (298450, 24), f"unexpected shape {df.shape}"

# GUARD 2: id is unique (primary key sanity)
assert df["id"].is_unique, "id not unique!"

# GUARD 3: lat/lon parse to float and sit inside Bengaluru bbox (allow tiny margin)
lat = pd.to_numeric(df["latitude"], errors="coerce")
lon = pd.to_numeric(df["longitude"], errors="coerce")
inbox = lat.between(12.7, 13.35) & lon.between(77.3, 77.9)
print("coords parseable:", round(lat.notna().mean()*100, 4), "% ; in-bbox:", round(inbox.mean()*100, 4), "%")

# Persist an immutable raw checkpoint as parquet (fast reload on token expiry)
os.makedirs("/kaggle/working", exist_ok=True)
df.to_parquet("/kaggle/working/raw.parquet")
print("checkpoint raw.parquet written")
print("DTYPES:", df.dtypes.astype(str).to_dict())
print("HEAD2:", json.dumps(df.head(2).to_dict(orient="records"), default=str)[:1500])


## `10_nullmap.py`

In [ ]:
import pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
df = pd.read_parquet("/kaggle/working/raw.parquet")
NULLTOK = df.isin(["NULL","null","",None]) | df.isna()
nullpct = (NULLTOK.mean()*100).sort_values(ascending=False)
print(nullpct.round(2).to_string())

# GUARD: known-dead columns are 100% null (from local profiling)
for c in ["description","closed_datetime","action_taken_timestamp"]:
    assert round(nullpct[c],1) == 100.0, f"{c} not fully null: {nullpct[c]}"
print("GUARD OK: dead columns 100% null")

ax = nullpct.plot.barh(figsize=(8,9)); ax.invert_yaxis()
ax.set_title("Null % by column"); ax.set_xlabel("% null")
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/10_nullmap.png", dpi=120)
print("saved 10_nullmap.png")


## `11_parse_arrays.py`

In [ ]:
import os; os.makedirs("/kaggle/working/derived", exist_ok=True)
import pandas as pd, json
df = pd.read_parquet("/kaggle/working/raw.parquet")
def parse(x):
    try: return json.loads(x) if isinstance(x,str) else []
    except Exception: return None
viol = df["violation_type"].map(parse)
offc = df["offence_code"].map(parse)

# GUARD 1: every row parses (no None)
assert viol.isna().sum()==0 and offc.isna().sum()==0, "unparseable array rows exist"
# GUARD 2: parallel arrays have equal length row-wise
mismatch = (viol.map(len) != offc.map(len))
print("length-mismatch rows:", int(mismatch.sum()))
assert mismatch.sum()==0, "violation_type and offence_code lengths differ"

# Build the code<->label map and check it's 1:1
pairs = set()
for vs,os_ in zip(viol,offc):
    for v,o in zip(vs,os_): pairs.add((o,v))
code2label = {}; ambig = []
for o,v in sorted(pairs):
    if o in code2label and code2label[o]!=v: ambig.append((o,v,code2label[o]))
    code2label[o]=v
print("distinct offence codes:", len(code2label), "| ambiguous mappings:", ambig)
print("CODE->LABEL:")
for o in sorted(code2label): print(f"  {o}: {code2label[o]}")
df_arr = df[["id"]].copy(); df_arr["viol"]=viol; df_arr["offc"]=offc
df_arr["n_violations"]=viol.map(len)
df_arr.to_parquet("/kaggle/working/derived/arrays.parquet")
print("n_violations distribution:")
print(df_arr["n_violations"].value_counts().sort_index().to_string())
print("rows with >1 violation: %.2f%%" % ((df_arr["n_violations"]>1).mean()*100))


## `12_validation.py`

In [ ]:
import pandas as pd
df = pd.read_parquet("/kaggle/working/raw.parquet")
isnull = lambda s: df[s].isin(["NULL",""]) | df[s].isna()
vs = df["validation_status"].where(~isnull("validation_status"), "NULL")
print("validation_status counts:")
print(vs.value_counts(dropna=False).to_string())
print("share:", (vs.value_counts(normalize=True)*100).round(2).to_dict())

# GUARD: the 42%-null block is co-null (one review event populates all four)
block = ["validation_status","updated_vehicle_number","updated_vehicle_type","validation_timestamp"]
conull = pd.DataFrame({c: isnull(c) for c in block})
print("\nco-null agreement matrix:")
print(conull.corr().round(3).to_string())

# scita send vs validation outcome
print("\nvalidation_status x data_sent_to_scita (row-normalised):")
print(pd.crosstab(vs, df["data_sent_to_scita"], normalize="index").round(3).to_string())

# is 'rejected'/'duplicate' spatially/station concentrated?
df["_vs"]=vs
bad = df["_vs"].isin(["rejected","duplicate"])
print("\nrejected+duplicate share: %.2f%%" % (bad.mean()*100))
print("top stations by rejected+duplicate RATE (min 1000 tickets):")
g = df.groupby("police_station")["_vs"].agg(lambda s: s.isin(["rejected","duplicate"]).mean())
cnt = df["police_station"].value_counts()
g = g[cnt[g.index]>=1000].sort_values(ascending=False)
print((g*100).round(1).head(10).to_string())


## `13_clean.py`

In [ ]:
import pandas as pd, numpy as np
df = pd.read_parquet("/kaggle/working/raw.parquet")
n0=len(df)
for c in ["latitude","longitude"]: df[c]=pd.to_numeric(df[c],errors="coerce")
df["created_dt"]=pd.to_datetime(df["created_datetime"],errors="coerce",utc=True)

# duplicate key: same vehicle, same ~location (5dp), same minute
df["_dupkey"]=(df["vehicle_number"].astype(str)+"|"+df["latitude"].round(5).astype(str)
              +"|"+df["longitude"].round(5).astype(str)+"|"+df["created_dt"].dt.floor("min").astype(str))
dups=df["_dupkey"].duplicated().sum()
print("exact-ish duplicate rows:", dups, f"({dups/n0*100:.2f}%)")
clean=df.drop_duplicates("_dupkey").copy()

# bbox filter (Gate 0 branch: expected ~0 dropped)
inbox=clean["latitude"].between(12.7,13.35)&clean["longitude"].between(77.3,77.9)
print("dropping out-of-bbox:", int((~inbox).sum()))
clean=clean[inbox]

# is_valid flag (Gate 1A): exclude rejected/duplicate; keep approved/created1/processing/NULL
isnull = clean["validation_status"].isin(["NULL",""]) | clean["validation_status"].isna()
vs = clean["validation_status"].where(~isnull, "NULL")
clean["validation_status_clean"]=vs
clean["is_valid"]= ~vs.isin(["rejected","duplicate"])
print("is_valid share: %.2f%%" % (clean["is_valid"].mean()*100))

clean=clean.drop(columns=["description","closed_datetime","action_taken_timestamp","_dupkey"])
# GUARD: monotonic shrink only, never grew, and lost < 5% total
assert len(clean)<=n0 and len(clean)>=0.95*n0, f"cleaning removed too much: {n0}->{len(clean)}"
clean.to_parquet("/kaggle/working/cleaned.parquet")
print("CLEANED rows:", len(clean), "from", n0, "| cols:", clean.shape[1])


## `14_enforcement_bias.py`

In [ ]:
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.close("all")   # guard: persistent kernel retains figure state across cells
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
def gini(counts):
    x=np.sort(np.asarray(counts,float)); n=len(x); cum=np.cumsum(x)
    return (n+1-2*np.sum(cum)/cum[-1])/n
for col in ["device_id","created_by_id","police_station"]:
    c=df[col].value_counts()
    share_top10=c.head(max(1,int(len(c)*0.1))).sum()/c.sum()
    print(f"{col}: {len(c)} unique | top-10% make {share_top10*100:.1f}% of tickets | Gini={gini(c.values):.3f}")
fig,ax=plt.subplots(figsize=(11,4))
top=df["device_id"].value_counts().head(30)
ax.bar(range(len(top)), top.values)
ax.set_xticks(range(len(top))); ax.set_xticklabels(top.index, rotation=90, fontsize=7)
ax.set_ylabel("tickets"); ax.set_xlabel("device_id")
ax.set_title("Tickets per device (top 30 of %d)"%df['device_id'].nunique())
plt.tight_layout()
plt.savefig("/kaggle/working/eda_out/14_device_concentration.png",dpi=120)
print("saved 14_device_concentration.png")
# how many devices/officers produce 80% of tickets?
for col in ["device_id","created_by_id"]:
    c=df[col].value_counts().sort_values(ascending=False)
    k=(c.cumsum()<=0.8*c.sum()).sum()+1
    print(f"{col}: {k} of {len(c)} ({k/len(c)*100:.1f}%) produce 80% of tickets")


## `20_univariate.py`

In [ ]:
import pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, json
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
viol=df["violation_type"].map(lambda x: json.loads(x) if isinstance(x,str) else [])
exploded=viol.explode()
# GUARD: explode conserves total tags
assert exploded.notna().sum()==viol.map(len).sum(), "explode lost tags"
print("GUARD OK: tag conservation")
fig,axes=plt.subplots(2,2,figsize=(16,12))
df["vehicle_type"].value_counts().head(15).plot.bar(ax=axes[0,0],title="vehicle_type (top15)")
exploded.value_counts().head(15).plot.bar(ax=axes[0,1],title="violation_type (top15)")
df["police_station"].value_counts().head(15).plot.bar(ax=axes[1,0],title="police_station (top15)")
df["center_code"].value_counts().head(15).plot.bar(ax=axes[1,1],title="center_code (top15)")
for ax in axes.flat: ax.tick_params(axis="x",labelsize=7,rotation=90)
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/20_univariate.png",dpi=110)
print("saved 20_univariate.png")
print("\nvehicle_type top:", df["vehicle_type"].value_counts().head(8).to_dict())
print("violation top:", exploded.value_counts().head(8).to_dict())


## `21_cooccurrence.py`

In [ ]:
import pandas as pd, numpy as np, json, itertools, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
viol=df["violation_type"].map(lambda x: json.loads(x) if isinstance(x,str) else [])
labels=sorted({v for vs in viol for v in vs})
idx={l:i for i,l in enumerate(labels)}
M=np.zeros((len(labels),len(labels)),int)
for vs in viol:
    for a,b in itertools.combinations(set(vs),2):
        M[idx[a],idx[b]]+=1; M[idx[b],idx[a]]+=1
top=pd.Series([v for vs in viol for v in vs]).value_counts().head(12).index.tolist()
sub=[idx[t] for t in top]
plt.figure(figsize=(12,10))
sns.heatmap(pd.DataFrame(M[np.ix_(sub,sub)],index=top,columns=top),annot=True,fmt="d",cmap="rocket_r")
plt.title("Violation co-occurrence (top 12, same ticket)"); plt.tight_layout()
plt.savefig("/kaggle/working/eda_out/21_cooccurrence.png",dpi=110)
print("saved 21_cooccurrence.png")
# strongest pairs
pairs=[]
for a,b in itertools.combinations(top,2):
    pairs.append((M[idx[a],idx[b]],a,b))
print("top co-occurring pairs:")
for c,a,b in sorted(pairs,reverse=True)[:8]: print(f"  {c:6d}  {a}  +  {b}")


## `22_severity.py`

In [ ]:
import pandas as pd, json
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
TIER={
 # Tier 3 - blocks moving lane / intersection
 "PARKING IN A MAIN ROAD":3,"DOUBLE PARKING":3,"PARKING NEAR ROAD CROSSING":3,
 "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS":3,"H T V PROHIBITED":3,
 "AGAINST ONE WAY/NO ENTRY":3,"STOPING ON WHITE/STOP LINE":3,"U TURN PROHIBITED":3,
 # Tier 2 - narrows carriageway / edge
 "WRONG PARKING":2,"NO PARKING":2,"PARKING OPPOSITE TO ANOTHER PARKED VEHICLE":2,
 "PARKING OTHER THAN BUS STOP":2,"PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC":2,
 # Tier 1 - footpath
 "PARKING ON FOOTPATH":1,
 # Tier 0 - everything else (document/behaviour/moving) defaults to 0
}
viol=df["violation_type"].map(lambda x: json.loads(x) if isinstance(x,str) else [])
alllabels={v for vs in viol for v in vs}
unmapped=sorted(alllabels-set(TIER))
print("labels defaulting to Tier 0:", unmapped)
# GUARD: no Tier-2/3 label accidentally unmapped (these are the only acceptable Tier-0 defaults)
EXPECTED_T0={"DEFECTIVE NUMBER PLATE","USING BLACK FILM/OTHER MATERIALS","WITHOUT SIDE MIRROR",
 "REFUSE TO GO FOR HIRE","DEMANDING EXCESS FARE","FAIL TO USE SAFETY BELTS","RIDER NOT WEARING HELMET",
 "2W/3W - USING MOBILE PHONE","OTHER - USING MOBILE PHONE","JUMPING TRAFFIC SIGNAL",
 "VIOLATING LANE DISIPLINE","OBSTRUCTING DRIVER","CARRYING LENGHTY MATERIAL"}
assert set(unmapped)<=EXPECTED_T0, f"unexpected unmapped (would under-score!): {set(unmapped)-EXPECTED_T0}"
print("GUARD OK: only known Tier-0 labels are unmapped")
sev=viol.map(lambda vs:[TIER.get(v,0) for v in vs])
out=df[["id","is_valid"]].copy()
out["max_sev"]=sev.map(lambda s:max(s) if s else 0)
out["sev_sum"]=sev.map(sum)
out.to_parquet("/kaggle/working/derived/severity.parquet")
print("max_sev distribution:")
print(out["max_sev"].value_counts().sort_index().to_string())
print("tickets with a Tier-3 violation: %.2f%%" % ((out["max_sev"]==3).mean()*100))


## `30_time_anomaly.py`

In [ ]:
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
ist=pd.to_datetime(df["created_dt"]).dt.tz_convert("Asia/Kolkata")
df["hour"]=ist.dt.hour
fig,ax=plt.subplots(1,2,figsize=(16,5))
df["hour"].value_counts().sort_index().plot.bar(ax=ax[0],title="Hour-of-day (IST) — overall")
ax[0].set_xlabel("hour"); ax[0].set_ylabel("tickets")
top6=df["police_station"].value_counts().head(6).index
for s in top6:
    df[df.police_station==s]["hour"].value_counts(normalize=True).sort_index().reindex(range(24),fill_value=0).plot(ax=ax[1],label=s,marker=".")
ax[1].legend(fontsize=7); ax[1].set_title("Hour profile by station (normalised)"); ax[1].set_xlabel("hour")
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/30_time_anomaly.png",dpi=110)
print("saved 30_time_anomaly.png")
dead=df["hour"].between(15,21).mean()*100
print(f"share of tickets in 15:00-21:00 IST: {dead:.2f}%")
# uniformity test: std of per-station dead-window share
per_station_dead=df.groupby("police_station")["hour"].apply(lambda h:h.between(15,21).mean())
print("per-station dead-share: mean=%.3f std=%.4f min=%.3f max=%.3f"%(
    per_station_dead.mean(),per_station_dead.std(),per_station_dead.min(),per_station_dead.max()))
# also check per-device uniformity (top 50 devices)
topdev=df["device_id"].value_counts().head(50).index
pdd=df[df.device_id.isin(topdev)].groupby("device_id")["hour"].apply(lambda h:h.between(15,21).mean())
print("per-device(top50) dead-share: mean=%.3f std=%.4f"%(pdd.mean(),pdd.std()))
# created_dt vs modified lag
mod=pd.to_datetime(df["modified_datetime"],errors="coerce",utc=True)
lag=(mod-df["created_dt"]).dt.total_seconds()/3600
print("created->modified lag hrs: median=%.2f p90=%.2f"%(lag.median(),lag.quantile(.9)))


## `31_calendar.py`

In [ ]:
import pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
ist=pd.to_datetime(df["created_dt"]).dt.tz_convert("Asia/Kolkata")
daily=ist.dt.date.value_counts().sort_index()
# GUARD: date range matches expectation (Nov-09 2023 .. ~Apr-09 2024)
assert str(daily.index.min())>="2023-11-08" and str(daily.index.max())<="2024-04-10", \
    f"date range drift {daily.index.min()}..{daily.index.max()}"
print("GUARD OK date range:", daily.index.min(), "->", daily.index.max())
fig,ax=plt.subplots(2,1,figsize=(15,10))
s=pd.Series(daily.values,index=pd.to_datetime(list(daily.index)))
s.plot(ax=ax[0],alpha=.4,label="daily"); s.rolling(7).mean().plot(ax=ax[0],lw=2,label="7d rolling")
ax[0].set_title("Daily tickets + 7d rolling mean"); ax[0].legend()
dow=ist.dt.day_name().value_counts().reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
dow.plot.bar(ax=ax[1],title="Tickets by day-of-week"); ax[1].tick_params(rotation=0)
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/31_calendar.png",dpi=110)
print("saved 31_calendar.png")
print("monthly totals:")
print(ist.dt.to_period("M").value_counts().sort_index().to_string())
print("day-of-week:", dow.to_dict())


## `40_grid.py`

In [ ]:
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
df["glat"]=df["latitude"].round(3); df["glon"]=df["longitude"].round(3)  # ~110m cells
grid=df.groupby(["glat","glon"]).size().rename("n").reset_index()
# GUARD: binning conserves all rows
assert grid["n"].sum()==len(df), "grid lost rows"
print("GUARD OK: %d rows -> %d cells"%(len(df),len(grid)))
grid.to_parquet("/kaggle/working/derived/grid_counts.parquet")
gs=grid.sort_values("n",ascending=False)
cells_50=(gs["n"].cumsum()<=0.5*len(df)).sum()+1
print("top cell count:", int(grid["n"].max()),
      "| cells holding 50%% of tickets: %d (%.2f%% of cells)"%(cells_50,cells_50/len(grid)*100))
print("top 10 cells:")
print(gs.head(10).to_string(index=False))
plt.figure(figsize=(10,10))
hb=plt.hexbin(df["longitude"],df["latitude"],gridsize=120,cmap="inferno",bins="log")
plt.colorbar(hb,label="log10(tickets)"); plt.title("Parking-violation density — Bengaluru")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.savefig("/kaggle/working/eda_out/40_density.png",dpi=120)
print("saved 40_density.png")


## `41_dbscan.py`

In [ ]:
import pandas as pd, numpy as np
from sklearn.cluster import DBSCAN
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
sev=pd.read_parquet("/kaggle/working/derived/severity.parquet")
df=df.merge(sev[["id","max_sev","sev_sum"]],on="id",how="left")
coords=np.radians(df[["latitude","longitude"]].values)
eps=150/6371000.0  # 150m in radians (haversine)
db=DBSCAN(eps=eps,min_samples=30,metric="haversine",algorithm="ball_tree").fit(coords)
df["cluster"]=db.labels_
n_clusters=int(df["cluster"].nunique()-(1 if -1 in df["cluster"].values else 0))
noise=(df["cluster"]==-1).mean()*100
print("clusters: %d | noise: %.1f%%"%(n_clusters,noise))
agg=(df[df.cluster>=0].groupby("cluster")
     .agg(n=("id","size"),n_valid=("is_valid","sum"),
          lat=("latitude","mean"),lon=("longitude","mean"),
          tier3_share=("max_sev",lambda s:(s>=3).mean()),
          sev_sum=("sev_sum","sum"),
          top_station=("police_station",lambda s:s.mode().iat[0]))
     .sort_values("n",ascending=False))
agg.to_parquet("/kaggle/working/derived/hotspots.parquet")
print("top 20 hotspots by ticket count:")
print(agg.head(20).round(3).to_string())


## `42_map_junctions.py`

In [ ]:
import pandas as pd, folium
from folium.plugins import HeatMap
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
hot=pd.read_parquet("/kaggle/working/derived/hotspots.parquet")
m=folium.Map(location=[12.97,77.59],zoom_start=12,tiles="cartodbpositron")
samp=df[["latitude","longitude"]].sample(min(50000,len(df)),random_state=0).values.tolist()
HeatMap(samp,radius=8,blur=6).add_to(m)
for cl,r in hot.head(20).iterrows():
    folium.CircleMarker([r.lat,r.lon],radius=6,color="red",fill=True,fill_opacity=0.7,
        popup=f"cluster {cl}: {int(r.n)} tickets, T3={r.tier3_share:.0%}, {r.top_station}").add_to(m)
m.save("/kaggle/working/eda_out/42_hotspots.html")
print("saved 42_hotspots.html")
# junction linkage
named=df["junction_name"].ne("No Junction")&df["junction_name"].notna()
print("tickets at NAMED junctions: %.2f%%"%(named.mean()*100))
print("top 15 named junctions:")
print(df[named]["junction_name"].value_counts().head(15).to_string())
# do hotspot clusters sit on junctions?
dfj=df.copy()
print("\nshare-at-named-junction within top-8 clusters:")
import numpy as np
from sklearn.cluster import DBSCAN
# reuse cluster labels by recomputing quickly is costly; instead bucket by nearest hotspot center
# (lightweight: report overall named share by station for top hotspot stations)
for st in hot.head(8)["top_station"].unique():
    sub=df[df.police_station==st]
    print(f"  {st:18s} named-junction share = {sub['junction_name'].ne('No Junction').mean()*100:5.1f}%")


## `43_bias_normalised.py`

In [ ]:
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
df["glat"]=df["latitude"].round(3); df["glon"]=df["longitude"].round(3)
# per-cell: raw count, distinct devices, distinct officers
g=df.groupby(["glat","glon"]).agg(n=("id","size"),
                                  n_dev=("device_id","nunique"),
                                  n_off=("created_by_id","nunique")).reset_index()
# a genuine hotspot is seen by MANY devices; a patrol artifact is one device hammering a spot
g["tickets_per_device"]=g["n"]/g["n_dev"]
g=g[g["n"]>=50]  # focus on real cells
g["rank_raw"]=g["n"].rank(ascending=False)
g["rank_perdev"]=g["tickets_per_device"].rank(ascending=False)
g["rank_shift"]=g["rank_raw"]-g["rank_perdev"]   # +ve: looks smaller once normalised
print("cells with >=50 tickets:",len(g))
print("\nTOP 10 by RAW count:")
print(g.sort_values("n",ascending=False).head(10)[["glat","glon","n","n_dev","tickets_per_device"]].round(1).to_string(index=False))
print("\nTOP 10 by TICKETS-PER-DEVICE (single-device-dominated = likely patrol artifact):")
print(g.sort_values("tickets_per_device",ascending=False).head(10)[["glat","glon","n","n_dev","tickets_per_device"]].round(1).to_string(index=False))
print("\ncorrelation(raw count, distinct devices): %.3f"%g["n"].corr(g["n_dev"]))
g.to_parquet("/kaggle/working/derived/grid_bias.parquet")
fig,ax=plt.subplots(1,2,figsize=(15,7))
sc0=ax[0].scatter(g["glon"],g["glat"],c=np.log10(g["n"]),cmap="inferno",s=8)
ax[0].set_title("Raw ticket density (log)"); plt.colorbar(sc0,ax=ax[0])
sc1=ax[1].scatter(g["glon"],g["glat"],c=g["n_dev"],cmap="viridis",s=8)
ax[1].set_title("Distinct devices per cell (breadth)"); plt.colorbar(sc1,ax=ax[1])
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/43_bias_normalised.png",dpi=110)
print("saved 43_bias_normalised.png")


## `50_crosstabs.py`

In [ ]:
import pandas as pd, json, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
sev=pd.read_parquet("/kaggle/working/derived/severity.parquet")
df=df.merge(sev[["id","max_sev","sev_sum"]],on="id")
# vehicle x severity (use sev_sum bucket since max_sev is near-binary)
ct=pd.crosstab(df["vehicle_type"],df["max_sev"],normalize="index")
assert (ct.sum(axis=1).round(3)==1).all(), "crosstab rows don't normalise"
print("GUARD OK crosstab normalised")
topveh=df["vehicle_type"].value_counts().head(15).index
plt.figure(figsize=(9,8)); sns.heatmap(ct.loc[topveh],annot=True,fmt=".3f",cmap="mako")
plt.title("Vehicle type x max severity tier (row-normalised)"); plt.tight_layout()
plt.savefig("/kaggle/working/eda_out/50_vehicle_severity.png",dpi=110)
print("saved 50_vehicle_severity.png")
# which vehicle types most often cause Tier-3
print("\nTier-3 rate by vehicle (top by rate, min 500 tickets):")
g=df.groupby("vehicle_type").agg(n=("id","size"),t3=("max_sev",lambda s:(s>=3).mean()))
print((g[g.n>=500].sort_values("t3",ascending=False).assign(t3pct=lambda d:(d.t3*100).round(1))[["n","t3pct"]]).head(12).to_string())
# per-station mean severity ranking
print("\nstations by mean sev_sum (min 2000 tickets):")
sg=df.groupby("police_station").agg(n=("id","size"),msev=("sev_sum","mean"))
print(sg[sg.n>=2000].sort_values("msev",ascending=False).assign(msev=lambda d:d.msev.round(2)).head(12).to_string())


## `51_space_time.py`

In [ ]:
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
ist=pd.to_datetime(df["created_dt"]).dt.tz_convert("Asia/Kolkata")
df["hour"]=ist.dt.hour
# recompute clusters (cheap enough) to attach hour profiles to top hotspots
coords=np.radians(df[["latitude","longitude"]].values)
df["cluster"]=DBSCAN(eps=150/6371000.0,min_samples=30,metric="haversine",algorithm="ball_tree").fit(coords).labels_
top=df[df.cluster>=0]["cluster"].value_counts().head(8).index
plt.figure(figsize=(12,6))
for cl in top:
    sub=df[df.cluster==cl]
    prof=sub["hour"].value_counts(normalize=True).sort_index().reindex(range(24),fill_value=0)
    st=sub["police_station"].mode().iat[0]
    plt.plot(range(24),prof.values,marker=".",label=f"cl{cl} ({st}, n={len(sub)})")
plt.legend(fontsize=7); plt.xlabel("hour (IST)"); plt.ylabel("share of cluster's tickets")
plt.title("ENFORCEMENT-ACTIVITY hour profile by top-8 hotspot (NOT congestion timing)")
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/51_space_time.png",dpi=110)
print("saved 51_space_time.png")
# quantify how different the hotspots' peak enforcement hours are
for cl in top:
    sub=df[df.cluster==cl]
    pk=int(sub["hour"].value_counts().idxmax())
    print(f"cluster {int(cl):3d} ({sub['police_station'].mode().iat[0]:16s}) peak enforcement hour = {pk:02d}:00 IST, n={len(sub)}")


## `60_probes.py`

In [ ]:
import pandas as pd, numpy as np
df=pd.read_parquet("/kaggle/working/cleaned.parquet")

print("="*60,"\n6.1 REPEAT OFFENDERS")
vc=df["vehicle_number"].value_counts()
print("unique vehicles:",len(vc),"| max tickets on one vehicle:",int(vc.max()))
print("tickets-per-vehicle: mean=%.2f median=%d p99=%d"%(vc.mean(),vc.median(),vc.quantile(.99)))
print("vehicles with >=10 tickets:",int((vc>=10).sum()),"| their share of all tickets: %.1f%%"%(vc[vc>=10].sum()/vc.sum()*100))
rep=vc[vc>=10].index
sub=df[df.vehicle_number.isin(rep)]
# are repeat offenders spatially concentrated? distinct cells per repeat vehicle
sub=sub.assign(glat=sub.latitude.round(3),glon=sub.longitude.round(3))
cells_per=sub.groupby("vehicle_number").apply(lambda d:d.groupby(["glat","glon"]).ngroups,include_groups=False)
print("repeat offenders: median distinct 110m-cells they're ticketed in = %.1f"%cells_per.median())

print("="*60,"\n6.2 PLATE-CORRECTION SIGNAL")
rev=df[df["validation_status_clean"]!="NULL"]
chg=(rev["vehicle_number"].astype(str)!=rev["updated_vehicle_number"].astype(str))
print("reviewed tickets:",len(rev),"| vehicle_number changed on review: %.2f%%"%(chg.mean()*100))
tchg=(rev["vehicle_type"].astype(str)!=rev["updated_vehicle_type"].astype(str))
print("vehicle_TYPE changed on review: %.2f%%"%(tchg.mean()*100))

print("="*60,"\n6.3 data_sent_to_scita MEANING")
print("overall TRUE rate: %.1f%%"%((df["data_sent_to_scita"]=="TRUE").mean()*100))
print("TRUE rate by validation_status:")
print(df.groupby("validation_status_clean")["data_sent_to_scita"].apply(lambda s:(s=="TRUE").mean()).round(3).to_string())
print("TRUE rate by month:")
ist=pd.to_datetime(df["created_dt"]).dt.tz_convert("Asia/Kolkata")
print(df.assign(m=ist.dt.to_period("M").astype(str)).groupby("m")["data_sent_to_scita"].apply(lambda s:(s=="TRUE").mean()).round(3).to_string())


## `63_severity_density.py`

In [ ]:
import pandas as pd, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.close("all")
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
sev=pd.read_parquet("/kaggle/working/derived/severity.parquet")
df=df.merge(sev[["id","sev_sum","max_sev"]],on="id")
df["glat"]=df["latitude"].round(3); df["glon"]=df["longitude"].round(3)
g=df.groupby(["glat","glon"]).agg(n=("id","size"),impact=("sev_sum","sum"),
                                  t3=("max_sev",lambda s:(s>=3).sum())).reset_index()
g=g[g["n"]>=50].copy()
g["rank_count"]=g["n"].rank(ascending=False)
g["rank_impact"]=g["impact"].rank(ascending=False)
g["divergence"]=g["rank_count"]-g["rank_impact"]   # +ve => impact rank BETTER than count rank (under-counted, high-impact)
g.to_parquet("/kaggle/working/derived/divergence.parquet")
print("cells>=50:",len(g),"| corr(count,impact): %.3f"%g["n"].corr(g["impact"]))
print("\nHIGH-IMPACT-but-LOWER-VOLUME cells (impact rank much better than count rank):")
hi=g.sort_values("divergence",ascending=False).head(10)
for _,r in hi.iterrows():
    st=df[(df.glat==r.glat)&(df.glon==r.glon)]["police_station"].mode().iat[0]
    print(f"  ({r.glat:.3f},{r.glon:.3f}) {st:16s} count#{int(r.rank_count):4d} impact#{int(r.rank_impact):4d}  n={int(r.n)} T3={int(r.t3)}")
fig,ax=plt.subplots(1,2,figsize=(15,7))
s0=ax[0].scatter(g.glon,g.glat,c=np.log10(g.n),cmap="inferno",s=10); ax[0].set_title("Volume (log tickets)"); plt.colorbar(s0,ax=ax[0])
s1=ax[1].scatter(g.glon,g.glat,c=np.log10(g.impact),cmap="inferno",s=10); ax[1].set_title("Congestion IMPACT (log severity-sum)"); plt.colorbar(s1,ax=ax[1])
plt.tight_layout(); plt.savefig("/kaggle/working/eda_out/63_severity_density.png",dpi=110)
print("saved 63_severity_density.png")


## `64_completeness.py`

In [ ]:
import pandas as pd, numpy as np
df=pd.read_parquet("/kaggle/working/cleaned.parquet")

print("="*60,"\nCENTER_CODE vs POLICE_STATION mapping")
g=df.dropna(subset=["center_code"]).groupby("center_code")["police_station"].nunique()
print("center_codes mapping to exactly 1 station: %d / %d"%((g==1).sum(),len(g)))
g2=df.groupby("police_station")["center_code"].nunique()
print("stations mapping to exactly 1 center_code: %d / %d"%((g2==1).sum(),len(g2)))

print("="*60,"\nLOCATION free-text road-type mining (native FE candidate)")
loc=df["location"].fillna("").str.lower()
KW={"main road":"main road","cross":"cross road","metro":"metro","market":"market",
    "circle":"circle","junction":"junction","layout":"layout","nagar":"nagar",
    "flyover":"flyover","bridge":"bridge","station":"station","temple":"temple",
    "mall":"mall","hospital":"hospital","school":"school","road":"road (any)"}
print("keyword presence in location string:")
for k,lbl in KW.items():
    print(f"  {lbl:14s} {loc.str.contains(k,regex=False).mean()*100:5.1f}%")
print("location non-empty: %.1f%%, unique strings: %d"%((loc!="").mean()*100, loc[loc!=""].nunique()))

print("="*60,"\nUNEXAMINED COLUMN CHECK")
print("data_sent_to_scita_timestamp: 86% null -> only present for sent+timestamped; carries no extra signal beyond data_sent_to_scita flag (skip).")
print("modified_datetime: used only via create->modify lag in 3.1 (sufficient).")
print("\nCOMPLETENESS: all 24 columns examined or explicitly dropped/skipped with reason.")
